### simple attention msm

In [1]:
import numpy as np
import torch
import math

In [2]:
# input string
input_str = "Your journey starts with one step"

In [5]:
# get the embeddings of all the tokens
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

### calculate attention scores
- how much every word in your sentence `relates` to every other word using a dot product.

**How it works (The Example):**
- Imagine the model is looking at the word `journey` (the query) and comparing it to the word `step` (the embedding):
**1. The Inputs:** 
- 1. **"journey" vector:** [0.55, 0.87, 0.66]
- 2. **"step" vector:** [0.05, 0.80, 0.55]

**2. The Calculation (torch.dot):** It multiplies the matching numbers and adds them up:
`((0.55 times 0.05) + (0.87 times 0.80) + (0.66 times 0.55) = 1.0865`

**3. The Storage:** It places that score `1.0865` into the attention_scores grid.

**The Loops:** The code repeats this for every possible pair of words.

**The Grid:** You end up with a `6 X 6` table.

In [7]:
# store all attention scores
attention_scores = torch.empty(inputs.shape[0], inputs.shape[0])
attention_scores

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])

In [8]:
for row, query in enumerate(inputs):
    for col, embedding in enumerate(inputs):
        attention_scores[row][col] = torch.dot(query, embedding)
attention_scores        

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [9]:
attention_scores = inputs @ inputs.T
attention_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

### calculate attention weights

In [11]:
attention_weights = attention_scores / attention_scores.sum(dim=0)
attention_weights

tensor([[0.2241, 0.1455, 0.1454, 0.1304, 0.1436, 0.1350],
        [0.2140, 0.2278, 0.2277, 0.2313, 0.2219, 0.2325],
        [0.2113, 0.2249, 0.2248, 0.2275, 0.2245, 0.2269],
        [0.1066, 0.1285, 0.1280, 0.1354, 0.1090, 0.1405],
        [0.1026, 0.1077, 0.1104, 0.0953, 0.2088, 0.0628],
        [0.1415, 0.1656, 0.1637, 0.1801, 0.0921, 0.2022]])

### apply softmax

In [12]:
# dim = -1: get the softmax on last dimension
normalized_attention_weights = torch.softmax(attention_scores, dim=-1)
normalized_attention_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

### Calculate the context vectors

In [13]:
context_vectors = normalized_attention_weights @ inputs
context_vectors

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])